## 4. `raise` & Custom Exceptions

### `raise` - deliberately trigger an exception
You use `raise` to signal that something has gone wrong that the current code cannot handle.

```python
raise ValueError("age must be positive")   # raise with message
raise                                       # re-raise the current exception
raise RuntimeError("msg") from original_exc  # chained exception
```

### Custom exceptions - create your own
Custom exceptions make your errors **meaningful and catchable**. Always inherit from `Exception` (or a suitable subclass).

```python
class MyError(Exception):
    pass
```

### Why create custom exceptions?
| Benefit | Example |
|---------|--------|
| Self-documenting | `InsufficientFundsError` is clearer than `ValueError` |
| Fine-grained catching | Catch `PaymentError` without catching all `Exception` |
| Carry extra data | Store `amount`, `balance` on the exception object |
| Build exception hierarchies | `AppError → DatabaseError → ConnectionError` |

In [1]:
# Basic raise
def set_age(age):
    if not isinstance(age, int):
        raise TypeError(f"age must be int, got {type(age).__name__}")
    if age < 0 or age > 150:
        raise ValueError(f"age must be 0-150, got {age}")
    return age

for val in [25, -1, 200, "old"]:
    try:
        print(f"  set_age({val!r}) → {set_age(val)}")
    except (TypeError, ValueError) as e:
        print(f"  set_age({val!r}) → {type(e).__name__}: {e}")

  set_age(25) → 25
  set_age(-1) → ValueError: age must be 0-150, got -1
  set_age(200) → ValueError: age must be 0-150, got 200
  set_age('old') → TypeError: age must be int, got str


In [2]:
# Re-raise — handle partially then let it propagate
def load_data(path):
    try:
        with open(path) as f:
            return f.read()
    except FileNotFoundError:
        print(f"  [load_data] Logging: file not found — {path!r}")
        raise    # re-raise the original exception unchanged

try:
    load_data("missing.txt")
except FileNotFoundError as e:
    print(f"  Caller caught: {e}")

  [load_data] Logging: file not found — 'missing.txt'
  Caller caught: [Errno 2] No such file or directory: 'missing.txt'


In [3]:
# Exception chaining with 'raise ... from'
def connect_db(dsn):
    try:
        raise ConnectionRefusedError("port 5432 refused")
    except ConnectionRefusedError as e:
        raise RuntimeError("Database unavailable") from e

try:
    connect_db("postgres://localhost/mydb")
except RuntimeError as e:
    print(f"RuntimeError: {e}")
    print(f"  Caused by: {e.__cause__}")

RuntimeError: Database unavailable
  Caused by: port 5432 refused


In [4]:
# ---- Custom Exceptions ----

# Simple custom exception
class ValidationError(Exception):
    """Raised when input validation fails."""
    pass

# Custom exception with extra data
class InsufficientFundsError(Exception):
    """Raised when a withdrawal exceeds the account balance."""
    def __init__(self, amount, balance):
        self.amount  = amount
        self.balance = balance
        super().__init__(
            f"Cannot withdraw ₹{amount:,.0f} — "
            f"balance is only ₹{balance:,.0f}"
        )

class BankAccount:
    def __init__(self, owner, balance=0):
        self.owner   = owner
        self.balance = balance

    def deposit(self, amount):
        if amount <= 0:
            raise ValidationError("Deposit amount must be positive")
        self.balance += amount
        print(f"  Deposited ₹{amount:,} → balance: ₹{self.balance:,}")

    def withdraw(self, amount):
        if amount <= 0:
            raise ValidationError("Withdrawal amount must be positive")
        if amount > self.balance:
            raise InsufficientFundsError(amount, self.balance)
        self.balance -= amount
        print(f"  Withdrew  ₹{amount:,} → balance: ₹{self.balance:,}")

acc = BankAccount("Purvi", 5000)

for action, amt in [("deposit", 2000), ("withdraw", 3000),
                    ("withdraw", 6000), ("deposit", -100)]:
    try:
        getattr(acc, action)(amt)
    except InsufficientFundsError as e:
        print(f"  InsufficientFundsError: {e}")
        print(f"    tried: ₹{e.amount:,}  available: ₹{e.balance:,}")
    except ValidationError as e:
        print(f"  ValidationError: {e}")

  Deposited ₹2,000 → balance: ₹7,000
  Withdrew  ₹3,000 → balance: ₹4,000
  InsufficientFundsError: Cannot withdraw ₹6,000 — balance is only ₹4,000
    tried: ₹6,000  available: ₹4,000
  ValidationError: Deposit amount must be positive
